In [28]:
from transformers import pipeline
import pandas as pd
from tqdm import tqdm

In [29]:
sentiment_pipeline = pipeline(model="finiteautomata/bertweet-base-sentiment-analysis")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 40201.00it/s]
RobertaForSequenceClassification LOAD REPORT from: finiteautomata/bertweet-base-sentiment-analysis
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0


In [30]:
df=pd.read_excel("Amazon_Comments.xlsx")
df.head()

,ASIN,#Comments,Title,Comments,Review Date,Star,Name,Part
0,B000CBLWD6,1,Five Stars,Works great on my 05 Neon!,"May 1, 2015",5 star,Autolite 97062 Spark Plug Wire Set,Spark Plug
1,B000CBLWD6,2,"Exellent product, very satisfied","Fit perfectly, easy installation, very happy.","July 31, 2014",5 star,Autolite 97062 Spark Plug Wire Set,Spark Plug
2,B000CBLWD6,3,Five Stars,great price and exactly what i needed for my 2...,"July 21, 2014",5 star,Autolite 97062 Spark Plug Wire Set,Spark Plug
3,B000CILQ6C,1,Good for the money,They worked at least 7/8 Amazon sent. The 8th ...,"July 15, 2016",5 star,Autolite APP764 Double Platinum Automotive Rep...,Spark Plug
4,B000CILQ6C,2,... had iridium plugs in my Cobra and it ran l...,I had iridium plugs in my Cobra and it ran lik...,"April 18, 2016",5 star,Autolite APP764 Double Platinum Automotive Rep...,Spark Plug


In [31]:
df = df.dropna(subset=['Comments']).reset_index(drop=True)

In [32]:
for i in tqdm(range(len(df))):
    text=df["Comments"][i]
    df.loc[i, 'Length of text'] = len(text)
    if len(text) <130:
            df.loc[i, 'Sentiment'] = sentiment_pipeline(text)[0]['label']
            df.loc[i, 'Confidence'] = round(sentiment_pipeline(text)[0]['score'], 2)
    else:
        chunks = [text[j:j+130] for j in range(0, len(text), 130)]
        
        sentiments = []
        scores = []
        
        for chunk in chunks:
            result = sentiment_pipeline(chunk)[0]
            sentiments.append(result['label'])
            scores.append(result['score'])
        
        df.loc[i, 'Sentiment'] = max(set(sentiments), key=sentiments.count)
        df.loc[i, 'Confidence'] = round(sum(scores) / len(scores), 2)
            

100%|██████████| 3044/3044 [03:19<00:00, 15.23it/s]


In [33]:
df

,ASIN,#Comments,Title,Comments,Review Date,Star,Name,Part,Length of text,Sentiment,Confidence
0,B000CBLWD6,1,Five Stars,Works great on my 05 Neon!,"May 1, 2015",5 star,Autolite 97062 Spark Plug Wire Set,Spark Plug,26.0,POS,0.99
1,B000CBLWD6,2,"Exellent product, very satisfied","Fit perfectly, easy installation, very happy.","July 31, 2014",5 star,Autolite 97062 Spark Plug Wire Set,Spark Plug,45.0,POS,0.99
2,B000CBLWD6,3,Five Stars,great price and exactly what i needed for my 2...,"July 21, 2014",5 star,Autolite 97062 Spark Plug Wire Set,Spark Plug,61.0,POS,0.99
3,B000CILQ6C,1,Good for the money,They worked at least 7/8 Amazon sent. The 8th ...,"July 15, 2016",5 star,Autolite APP764 Double Platinum Automotive Rep...,Spark Plug,69.0,NEG,0.88
4,B000CILQ6C,2,... had iridium plugs in my Cobra and it ran l...,I had iridium plugs in my Cobra and it ran lik...,"April 18, 2016",5 star,Autolite APP764 Double Platinum Automotive Rep...,Spark Plug,109.0,NEU,0.62
...,...,...,...,...,...,...,...,...,...,...,...
3039,B000C02FEM,2,What pump to buy?,There are the super cheap China sourced pumps ...,"January 19, 2022",5 star,Carter Fuel Systems Electric Fuel Pump Module ...,Fuel Pump,557.0,NEU,0.85
3040,B000C02FEM,3,Five Stars,Great product at a great price,"February 3, 2018",5 star,Carter Fuel Systems Electric Fuel Pump Module ...,Fuel Pump,30.0,POS,0.99
3041,B000C02FEM,4,good price,didnt know this part was recondition but worki...,"February 18, 2019",4 star,Carter Fuel Systems Electric Fuel Pump Module ...,Fuel Pump,60.0,POS,0.99
3042,B000C02FEM,5,Four Stars,working well so far,"March 28, 2017",4 star,Carter Fuel Systems Electric Fuel Pump Module ...,Fuel Pump,19.0,POS,0.98


In [34]:
df.to_excel("Amazon_Comments_Sentiment.xlsx", index=False)